In [0]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Config
CATALOG = "oil_gas_demo"
GOLD_AGG = f"{CATALOG}.gold.daily_demand_agg"
GOLD_FEATURES = f"{CATALOG}.gold.demand_features"


class DemandFeatureBuilder:
    """Builds ML-ready features from gold daily aggregates using SQL."""

    def __init__(self, spark, source_table: str, target_table: str):
        self.spark = spark
        self.source_table = source_table
        self.target_table = target_table


    def write_feature_table(self):
        """Create or replace the feature table with lags, rolling avg, and calendar fields."""
        logger.info(f"Building features: {self.source_table} -> {self.target_table}")
        self.spark.sql(f"""
            CREATE OR REPLACE TABLE {self.target_table} AS
            SELECT * FROM (
                SELECT
                    transaction_date,
                    product_name,
                    destination_city,
                    total_demand,
                    avg_unit_price,
                    total_inventory,
                    dayofweek(transaction_date) AS day_of_week,
                    month(transaction_date) AS month,
                    ROUND(LAG(total_demand, 1) OVER (
                        PARTITION BY product_name, destination_city ORDER BY transaction_date
                    ), 2) AS demand_lag_1,
                    ROUND(LAG(total_demand, 7) OVER (
                        PARTITION BY product_name, destination_city ORDER BY transaction_date
                    ), 2) AS demand_lag_7,
                    ROUND(AVG(total_demand) OVER (
                        PARTITION BY product_name, destination_city
                        ORDER BY transaction_date
                        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
                    ), 2) AS rolling_avg_7
                FROM {self.source_table}
            )
            WHERE demand_lag_7 IS NOT NULL
              AND rolling_avg_7 IS NOT NULL
        """)
        count = self.spark.table(self.target_table).count()
        logger.info(f"Feature table written: {self.target_table} ({count:,} rows)")

    def run(self):
        """Build features -> write."""
        self.write_feature_table()


# Execute
builder = DemandFeatureBuilder(
    spark=spark,
    source_table=GOLD_AGG,
    target_table=GOLD_FEATURES,
)
builder.run()